# Kings HUDL game analyzer

Press the Play button on each box below, one at a time, top to bottom. There are only five. Step 4 does everything in one go: it reads the scoreboard, finds skaters, finds faceoffs and bench changes, and builds your review page.

Before you start: click **Runtime**, then **Change runtime type**, pick **T4 GPU**, and press Save. Without a graphics card this is very slow.

**No Google Drive needed.** Leave step 2 alone. Click the folder icon on the left, drag your game video into the panel, wait for the upload to finish, then run step 3. Step 3 prints the file name, size and length so you can check it is the right file.

Use Chrome. Safari blocks parts of Colab.

In [ ]:
#@title 1. Get the newest program
import os, subprocess
if not os.path.exists('/content/kings-hudl'):
    subprocess.run(['git','clone','-q','https://github.com/rlahay-cloud/kings-hudl','/content/kings-hudl'],check=True)
else:
    subprocess.run(['git','-C','/content/kings-hudl','pull','-q'],check=True)
!pip -q install ultralytics lapx scikit-learn scipy easyocr
import torch
print('Graphics card:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE. Change the runtime type to T4 GPU, or this will take days.')

In [ ]:
#@title 2. Google Drive (optional, leave it off if it gives errors)
USE_DRIVE = False  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive did not connect:', e)
        print('That is fine. Turn USE_DRIVE off and drag the video into the Files panel instead.')
else:
    print('Skipping Drive. Next, drag your game video into the Files panel on the left (folder icon).')

In [ ]:
#@title 3. Pick the game
#@markdown Leave GAME_FILE blank to use the video you dragged into the Files panel. If you put more than one video there, type the name of the one to use.
GAME_FILE = ""  #@param {type:"string"}
#@markdown Leave these at 0 to do the whole game. To test, use START 0 and END 300 (the first 5 minutes).
START_SECONDS = 0  #@param {type:"number"}
END_SECONDS = 0  #@param {type:"number"}
import os, glob, time, cv2
EXT = ('.mov', '.mp4', '.mkv', '.m4v', '.avi')
def find_videos():
    out = []
    for d in ('/content', '/content/drive/MyDrive/KingsHUDL games'):
        for f in glob.glob(d + '/*'):
            if f.lower().endswith(EXT) and os.path.isfile(f) and not f.startswith('/content/sample_data'):
                out.append(f)
    return sorted(out)
if GAME_FILE.strip():
    g = GAME_FILE.strip()
    options = [g, '/content/' + g, '/content/drive/MyDrive/' + g]
    src = next((o for o in options if os.path.isfile(o)), None)
    assert src, 'I cannot find "%s". Click the folder icon on the left and check the spelling.' % g
else:
    vids = find_videos()
    assert vids, 'I see no video yet. Drag your game video into the Files panel on the left (folder icon), wait for it to finish uploading, then press Play again.'
    assert len(vids) == 1, 'I see more than one video: ' + ', '.join(os.path.basename(v) for v in vids) + '. Type the one you want in GAME_FILE above.'
    src = vids[0]
# make sure the upload has finished: the size must stop changing
s1 = os.path.getsize(src); time.sleep(4); s2 = os.path.getsize(src)
assert s1 == s2, 'The file is still uploading (it is growing). Wait for the upload to finish and press Play again.'
cap = cv2.VideoCapture(src)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 0
mins = (cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps / 60) if fps else 0
assert w > 0, 'I could not open that file as a video. It may not have finished uploading.'
print('USING THIS FILE:', os.path.basename(src))
print('  size: {:,} bytes ({:.2f} GB)'.format(s2, s2 / 1e9))
print('  picture: %d by %d, %.0f frames a second, %.1f minutes long' % (w, h, fps, mins))
print('Check these against the same file on your computer: right click it, Properties (Windows) or Get Info (Mac).')
print('The size in bytes and the length should match. If they do, it is the right file.')
name = os.path.splitext(os.path.basename(src))[0]
drive_ok = os.path.isdir('/content/drive/MyDrive/KingsHUDL games')
OUT = ('/content/drive/MyDrive/KingsHUDL games/analysis_' if drive_ok else '/content/analysis_') + name
os.makedirs(OUT, exist_ok=True)
local = src
print('Results will go to:', OUT, '' if drive_ok else '(not saved in Drive, so download the review page at the end)')

In [ ]:
#@title 4. Analyze the game and build the review page (one go)
#@markdown It reads the video once, in order. It saves as it goes. If it stops, press Play again and it carries on.
#@markdown To test first, set START 0 and END 300 in step 3.
import sys
sys.path.insert(0, '/content/kings-hudl/analyzer')
import onepass, kings_analyzer as ka
RINK = '/content/kings-hudl/analyzer/rinks/ingredion_rink1.json'
onepass.run(local, OUT, RINK, step=1.0, start=START_SECONDS, end=(END_SECONDS or None), imgsz=1280)
page = ka.build(local, OUT, RINK)
print('Done. The review page is downloading now. It is also here:', page)
from google.colab import files
files.download(page)


In [ ]:
#@title 5. Save a small results file to send to Claude (optional)
#@markdown Only small text results, no video and no clear pictures. Useful to improve the program.
import zipfile
from google.colab import files
keep = ['scoreboard.jsonl', 'detections.jsonl', 'meta.json', 'analysis.json']
with zipfile.ZipFile('/content/kings_results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for n in keep:
        p = OUT + '/' + n
        if os.path.exists(p):
            zf.write(p, n)
files.download('/content/kings_results_small.zip')
